## Setup and Imports

In [9]:
# Standard Libriaries

import sys
import os
import warnings
warnings.filterwarnings('ignore')  #Suppress minor warnings for cleaner output


sys.path.append('../..')

# Data manipulation
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from utils.helpers import load_orders, set_style, save_figure

set_style()

print('All imports successful✓')
print(f'pandas version: {pd.__version__}')
print(f'numpy version: {np.__version__}')



All imports successful✓
pandas version: 2.2.3
numpy version: 1.26.4


## Load and Validate Data

In [11]:
df_raw = load_orders(cleaned=False) # never modified
df = df_raw.copy()                  # working copy - safe to transform

print('Dataset Overview')
print(f'Rows:  {len(df):,}')
print(f'Columns:  {len(df.columns):,}')
print(f'\nColumn names:')
for col in df.columns:
    print(f'  {col}')

Dataset Overview
Rows:  21,864
Columns:  12

Column names:
  USER_ID
  ORDER_ID
  PURCHASE_TS
  SHIP_TS
  PRODUCT_NAME
  PRODUCT_ID
  USD_PRICE
  PURCHASE_PLATFORM
  MARKETING_CHANNEL
  ACCOUNT_CREATION_METHOD
  COUNTRY_CODE
  REGION


In [12]:
df.head()

,USER_ID,ORDER_ID,PURCHASE_TS,SHIP_TS,PRODUCT_NAME,PRODUCT_ID,USD_PRICE,PURCHASE_PLATFORM,MARKETING_CHANNEL,ACCOUNT_CREATION_METHOD,COUNTRY_CODE,REGION
0,2c06175e,0001328c3c220830,2020-12-24 00:00:00,2020-12-13,Nintendo Switch,e682,168.00,website,affiliate,unknown,US,NaN
1,ee8e5bc2,0002af7a5c6100772,2020-10-01 00:00:00,2020-09-21,Nintendo Switch,e682,160.61,website,direct,desktop,DE,EMEA
2,9eb4efe0,0002b8350e167074,2020-04-21 00:00:00,2020-02-16,Nintendo Switch,8d0d,151.20,website,direct,desktop,US,NaN
3,cac7cbaf,0006d06b98385729,2020-04-07 00:00:00,2020-04-04,Sony PlayStation 5 Bundle,54ed,1132.82,website,direct,desktop,AU,APAC
4,6b0230bc,00097279a2f46150,2020-11-24 00:00:00,2020-08-02,Nintendo Switch,8d0d,33.89,website,direct,desktop,TR,EMEA


In [13]:
print('Data Types and Null Counts')
info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2)
})
print(info_df.to_string())

Data Types and Null Counts
                                  dtype  null_count  null_pct
USER_ID                          object           0      0.00
ORDER_ID                         object           0      0.00
PURCHASE_TS                      object           0      0.00
SHIP_TS                  datetime64[ns]           0      0.00
PRODUCT_NAME                     object           0      0.00
PRODUCT_ID                       object           0      0.00
USD_PRICE                       float64           5      0.02
PURCHASE_PLATFORM                object           0      0.00
MARKETING_CHANNEL                object          83      0.38
ACCOUNT_CREATION_METHOD          object          83      0.38
COUNTRY_CODE                     object          38      0.17
REGION                           object       10531     48.17


In [14]:
# Parse the timestamp columns into proper datetime objects
# errors='coerce' means if a value cannot be parsed as a date,
# it becomes NaT (Not a Time) rather than crashing the whole operation

df['PURCHASE_TS'] = pd.to_datetime(df['PURCHASE_TS'], errors='coerce')
df['SHIP_TS'] = pd.to_datetime(df['SHIP_TS'], errors='coerce')

# Calculate fulfilment days: ship date minus purchase date
# Positive = normal (shipped after purchase)
# Zero     = same-day dispatch
# Negative = ANOMALY (shipped before purchase — impossible in reality)
df['FULFILMENT_DAYS'] = (df['SHIP_TS'] - df['PURCHASE_TS']).dt.days

print('Timestamps parsed successfully ✓')
print(f"Purchase TS range: {df['PURCHASE_TS'].min()} → {df['PURCHASE_TS'].max()}")
print(f"Ship TS range:     {df['SHIP_TS'].min()} → {df['SHIP_TS'].max()}")

Timestamps parsed successfully ✓
Purchase TS range: 2019-01-01 00:00:00 → 2021-02-28 00:00:00
Ship TS range:     2018-10-18 00:00:00 → 2021-11-16 00:00:00
